In [3]:
import torch
print("PyTorch 版本：", torch.__version__)
print("CUDA 可用：", torch.cuda.is_available())
print("Compute Capability：", torch.cuda.get_device_capability(0))

x = torch.rand(1000, 1000).cuda()
y = torch.mm(x, x)
print("GPU 運算測試成功！")

PyTorch 版本： 2.11.0+cu128
CUDA 可用： True
Compute Capability： (12, 0)
GPU 運算測試成功！


In [1]:
import urllib.request
import os

os.makedirs('model', exist_ok=True)

# 下載 Real-ESRGAN 模型
print("下載 RealESRGAN_x2plus.pth...")
urllib.request.urlretrieve(
    "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth",
    "model/RealESRGAN_x2plus.pth"
)

# 下載 GFPGAN 模型
print("下載 GFPGANv1.4.pth...")
urllib.request.urlretrieve(
    "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth",
    "model/GFPGANv1.4.pth"
)

print("兩個模型都下載完成！")
print("RealESRGAN 大小：", os.path.getsize("model/RealESRGAN_x2plus.pth"), "bytes")
print("GFPGAN 大小：", os.path.getsize("model/GFPGANv1.4.pth"), "bytes")

下載 RealESRGAN_x2plus.pth...
下載 GFPGANv1.4.pth...
兩個模型都下載完成！
RealESRGAN 大小： 67061725 bytes
GFPGAN 大小： 348632874 bytes


In [3]:
import glob

# 找到 basicsr 的 degradations.py
matches = glob.glob(r"C:\Users\USER\anaconda3\envs\idol_gpu\Lib\site-packages\basicsr\data\degradations.py")
print("找到檔案：", matches)

with open(matches[0], 'r') as f:
    content = f.read()

content = content.replace(
    'torchvision.transforms.functional_tensor',
    'torchvision.transforms.functional'
)

with open(matches[0], 'w') as f:
    f.write(content)

print("修正完成！")

找到檔案： ['C:\\Users\\USER\\anaconda3\\envs\\idol_gpu\\Lib\\site-packages\\basicsr\\data\\degradations.py']
修正完成！


In [2]:
import os
import cv2
import torch
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from gfpgan import GFPGANer

# 全域變數，避免每次都重新載入模型（很耗時）
_upsampler = None
_face_enhancer = None


def _load_models():
    """載入 Real-ESRGAN 和 GFPGAN 模型（只載入一次）"""
    global _upsampler, _face_enhancer

    if _upsampler is None:
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                        num_block=23, num_grow_ch=32, scale=2)
        _upsampler = RealESRGANer(
            scale=2,
            model_path='model/RealESRGAN_x2plus.pth',
            model=model,
            tile=0,
            tile_pad=10,
            pre_pad=0,
            half=True  # GPU 用 FP16 加速
        )
        print("Real-ESRGAN 載入完成")

    if _face_enhancer is None:
        _face_enhancer = GFPGANer(
            model_path='model/GFPGANv1.4.pth',
            upscale=2,
            arch='clean',
            channel_multiplier=2,
            bg_upsampler=_upsampler
        )
        print("GFPGAN 載入完成")


def enhance_video_hq(input_path, output_path, weight=0.3):
    """
    高品質影片增強：Real-ESRGAN（整體）+ GFPGAN（臉部）
    weight: GFPGAN 忠實度，越低越像本人（建議 0.3）
    """
    _load_models()

    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 輸出尺寸是原本的 2 倍
    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w * 2, h * 2)
    )

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # GFPGAN 處理（內含 Real-ESRGAN 做背景）
        _, _, output_frame = _face_enhancer.enhance(
            frame,
            has_aligned=False,
            only_center_face=False,
            paste_back=True,
            weight=weight
        )

        out.write(output_frame)
        frame_count += 1
        print(f"進度：{frame_count}/{total_frames}")

    cap.release()
    out.release()
    print(f"完成！輸出：{output_path}")
    return output_path


if __name__ == '__main__':
    # 測試用
    enhance_video_hq(
        input_path='0528.mp4',
        output_path='outputs/test_hq.mp4',
        weight=0.3
    )

Real-ESRGAN 載入完成


c:\Users\USER\anaconda3\envs\idol_gpu\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\USER\anaconda3\envs\idol_gpu\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


GFPGAN 載入完成
進度：1/129
進度：2/129
進度：3/129
進度：4/129
進度：5/129
進度：6/129
進度：7/129
進度：8/129
進度：9/129
進度：10/129
進度：11/129
進度：12/129
進度：13/129
進度：14/129
進度：15/129
進度：16/129
進度：17/129
進度：18/129
進度：19/129
進度：20/129
進度：21/129
進度：22/129
進度：23/129
進度：24/129
進度：25/129
進度：26/129
進度：27/129
進度：28/129
進度：29/129
進度：30/129
進度：31/129
進度：32/129
進度：33/129
進度：34/129
進度：35/129
進度：36/129
進度：37/129
進度：38/129
進度：39/129
進度：40/129
進度：41/129
進度：42/129
進度：43/129
進度：44/129
進度：45/129
進度：46/129
進度：47/129
進度：48/129
進度：49/129
進度：50/129
進度：51/129
進度：52/129
進度：53/129
進度：54/129
進度：55/129
進度：56/129
進度：57/129
進度：58/129
進度：59/129
進度：60/129
進度：61/129
進度：62/129
進度：63/129
進度：64/129
進度：65/129
進度：66/129
進度：67/129
進度：68/129
進度：69/129
進度：70/129
進度：71/129
進度：72/129
進度：73/129
進度：74/129
進度：75/129
進度：76/129
進度：77/129
進度：78/129
進度：79/129
進度：80/129
進度：81/129
進度：82/129
進度：83/129
進度：84/129
進度：85/129
進度：86/129
進度：87/129
進度：88/129
進度：89/129
進度：90/129
進度：91/129
進度：92/129
進度：93/129
進度：94/129
進度：95/129
進度：96/129
進度：97/129
進度：98/129
進度：99/129
進度：100/

In [ ]:
import os
import cv2
import torch
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
from gfpgan import GFPGANer

# 全域變數，避免每次重新載入模型
_upsampler = None
_face_enhancer = None


def _load_models():
    """載入 Real-ESRGAN 和 GFPGAN（只載入一次）"""
    global _upsampler, _face_enhancer

    if _upsampler is None:
        model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                        num_block=23, num_grow_ch=32, scale=2)
        _upsampler = RealESRGANer(
            scale=2,
            model_path='model/RealESRGAN_x2plus.pth',
            model=model,
            tile=0,
            tile_pad=10,
            pre_pad=0,
            half=True
        )
        print("Real-ESRGAN 載入完成")

    if _face_enhancer is None:
        # GFPGAN 只修臉，不放大（因為第一段已經放大），所以 upscale=1
        _face_enhancer = GFPGANer(
            model_path='model/GFPGANv1.4.pth',
            upscale=1,
            arch='clean',
            channel_multiplier=2,
            bg_upsampler=None  # 兩段式：背景不在這裡放大
        )
        print("GFPGAN 載入完成")


def enhance_video_hq(input_path, output_path, weight=0.3):
    """
    兩段式高品質影片增強：
    第一段 Real-ESRGAN 整體放大 2 倍
    第二段 GFPGAN 修臉（weight=0.3 低幻覺）
    """
    _load_models()

    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # 輸出尺寸是原本的 2 倍
    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (w * 2, h * 2)
    )

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # === 第一段：Real-ESRGAN 整體放大 2 倍 ===
        upscaled, _ = _upsampler.enhance(frame, outscale=2)

        # === 第二段：GFPGAN 修臉（不再放大）===
        _, _, restored = _face_enhancer.enhance(
            upscaled,
            has_aligned=False,
            only_center_face=False,
            paste_back=True,
            weight=weight
        )

        out.write(restored)
        frame_count += 1
        print(f"進度：{frame_count}/{total_frames}")

    cap.release()
    out.release()
    print(f"完成！輸出：{output_path}")
    return output_path


if __name__ == '__main__':
    enhance_video_hq(
        input_path='test_input.mp4',
        output_path='outputs/test_hq.mp4',
        weight=0.3
    )

Real-ESRGAN 載入完成
GFPGAN 載入完成
完成！輸出：outputs/test_hq.mp4


In [1]:
import os
os.chdir(r'C:\Users\USER\Desktop\image_final')

from video_enhance_hq import enhance_video_hq

enhance_video_hq(
    input_path='0528.mp4',
    output_path='outputs/test_1stage.mp4',
    weight=0.3
)

ModuleNotFoundError: No module named 'imageio_ffmpeg'

In [ ]:
import cv2

# 讀取你的高清影片
input_video = 'test_input.mp4'  # 換成你的檔名
cap = cv2.VideoCapture(input_video)
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# 輸出降質版（解析度縮到 1/3，模擬低清）
out = cv2.VideoWriter('test_lowres.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'),
                      fps, (w, h))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    # 先縮小再放大，製造模糊+壓縮感
    small = cv2.resize(frame, (w//3, h//3))
    blurry = cv2.resize(small, (w, h))
    out.write(blurry)

cap.release()
out.release()
print("降質版完成！檔名：test_lowres.mp4")

In [2]:
import cv2

for f in ['0528.2.mp4', 'outputs/test_hq2.mp4']:
    cap = cv2.VideoCapture(f)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f"{f}：{w}x{h}")

0528.2.mp4：854x480
outputs/test_hq2.mp4：1708x960


In [1]:
import torch; print(torch.__version__)

2.11.0+cu128
